# Surg-SegFormer on the full SAR-RARP50 dataset

Select a GPU runtime, then run each cell. This notebook downloads the official [training](https://rdr.ucl.ac.uk/articles/dataset/SAR-RARP50_train_set/24932529) and [test](https://rdr.ucl.ac.uk/articles/dataset/SAR-RARP50_test_set/24932499) archives directly to `My Drive/SAR-RARP50/archives`. Downloads survive runtime resets and resume when this cell is run again.

Allow about **32 GB of free Drive space** for the 54 archives. Operations 1-40 form the development data; four complete operations are held out for validation, and the official operations 41-50 remain test-only.

In [ ]:
!nvidia-smi
!git clone --branch 002-surg-segformer-dual-branch --single-branch https://github.com/alireza1420/SAR-RARP50-YOLOV8.git /content/surgery
%cd /content/surgery/surgformer
!python -m pip install -q -r requirements.txt

## Download the official archives directly to Drive

This cell is safe to rerun. A complete archive is skipped; an interrupted `.part` file resumes from its current size.

In [ ]:
from google.colab import drive
from pathlib import Path
import re
import requests
from tqdm.auto import tqdm

drive.mount('/content/drive')
ARCHIVE_ROOT = Path('/content/drive/MyDrive/SAR-RARP50/archives')
ARTICLES = {'train': 24932529, 'test': 24932499}
EXPECTED_ARCHIVES = {'train': 44, 'test': 10}
VIDEO_ARCHIVE = re.compile(r'video_\d{2}(?:_[12])?\.zip')

def download(file_info, directory):
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / file_info['name']
    partial = target.with_suffix(target.suffix + '.part')
    expected = int(file_info['size'])

    if target.exists() and target.stat().st_size == expected:
        print(f'skip {target.name}')
        return target
    if target.exists():
        target.replace(partial)
    if partial.exists() and partial.stat().st_size == expected:
        partial.replace(target)
        return target
    if partial.exists() and partial.stat().st_size > expected:
        partial.unlink()

    start = partial.stat().st_size if partial.exists() else 0
    headers = {'Range': f'bytes={start}-'} if start else {}
    with requests.get(file_info['download_url'], headers=headers, stream=True, timeout=(30, 300)) as response:
        if start and response.status_code != 206:
            start = 0
            mode = 'wb'
        else:
            mode = 'ab' if start else 'wb'
        response.raise_for_status()
        with partial.open(mode) as output, tqdm(
            total=expected, initial=start, unit='B', unit_scale=True, desc=target.name
        ) as progress:
            for chunk in response.iter_content(8 * 1024 * 1024):
                if chunk:
                    output.write(chunk)
                    progress.update(len(chunk))

    actual = partial.stat().st_size
    assert actual == expected, f'{target.name}: expected {expected} bytes, got {actual}'
    partial.replace(target)
    return target

downloaded = []
for split, article_id in ARTICLES.items():
    response = requests.get(f'https://api.figshare.com/v2/articles/{article_id}', timeout=60)
    response.raise_for_status()
    files = [file for file in response.json()['files'] if VIDEO_ARCHIVE.fullmatch(file['name'])]
    assert len(files) == EXPECTED_ARCHIVES[split], f'Unexpected {split} archive count: {len(files)}'
    for file_info in sorted(files, key=lambda item: item['name']):
        downloaded.append(download(file_info, ARCHIVE_ROOT / split))

assert len(downloaded) == 54
print(f'Ready: {len(downloaded)} archives, {sum(path.stat().st_size for path in downloaded) / 2**30:.1f} GiB')

## Extract and prepare all annotated frames

The archives stay in Drive. Extracted videos and prepared frames use the faster temporary Colab disk and are recreated after a runtime reset.

In [ ]:
import json
import shutil

RAW_ROOT = Path('/content/sar_rarp50_raw')
DATA_ROOT = Path('/content/sar_rarp50_prepared')
shutil.rmtree(RAW_ROOT, ignore_errors=True)
shutil.rmtree(DATA_ROOT, ignore_errors=True)
RAW_ROOT.mkdir(parents=True)

archives = sorted(ARCHIVE_ROOT.glob('*/*.zip'))
assert len(archives) == 54, f'Expected 54 archives, found {len(archives)}'
for index, archive in enumerate(archives, 1):
    print(f'[{index}/54] extracting {archive.name}')
    shutil.unpack_archive(archive, RAW_ROOT)

assert len(list(RAW_ROOT.rglob('video_left.avi'))) == 54
!python prepare_dataset.py /content/sar_rarp50_raw /content/sar_rarp50_prepared --val-operations 4 --seed 42

manifest = json.loads((DATA_ROOT / 'split_manifest.json').read_text())
assert manifest['frames']['train'] + manifest['frames']['val'] == 12998
assert manifest['frames']['test'] == 3252
manifest['frames'], manifest['operations']

## Configure and train

Every epoch visits every frame in the 36 training operations. `EPOCHS = 10` is a practical starting point for Colab; change it to `100` for the repository's original experiment setting if your compute budget permits.

In [ ]:
import torch
import yaml

EPOCHS = 10
config_path = Path('/content/surgery/surgformer/config.yaml')
config = yaml.safe_load(config_path.read_text())
task = config['data']['tasks']['sar_rarp50']
task['root'] = str(DATA_ROOT)
task['masks_root'] = str(DATA_ROOT / 'masks')

counts = manifest['class_pixel_counts']['train']
shares = manifest['class_pixel_share']['train']
common = [class_id for class_id in range(1, 10) if shares[class_id] >= 0.01]
rare = [class_id for class_id in range(1, 10) if shares[class_id] < 0.01]
assert common and rare and set(common + rare) == set(range(1, 10))
task['class_groups'] = {'coarse': [0, *common], 'fine': [0, *rare]}
task['untrainable_class_ids'] = [class_id for class_id in range(1, 10) if counts[class_id] == 0]
config['training']['batch_size'] = 1
config['training']['epochs'] = EPOCHS
config['training']['checkpoint_dir'] = '/content/drive/MyDrive/surgformer_results/full/checkpoints'
config_path.write_text(yaml.safe_dump(config, sort_keys=False))

assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime > Change runtime type > GPU'
print(torch.cuda.get_device_name(0))
print('coarse classes:', task['class_groups']['coarse'])
print('fine classes:', task['class_groups']['fine'])
print('untrainable classes:', task['untrainable_class_ids'])
!python test_prepare_dataset.py
!python smoke_test.py

In [ ]:
!python main.py --config config.yaml --task sar_rarp50